# A multifield, extensible map — what's *in* the cosmic web

The pipeline so far answers *where* the web is (void / sheet / filament / knot). This notebook makes the
map say *what's there* — and does it in a way that lets **new data layer in without a rewrite**.

The key fact: the CAMELS data is a **multifield** dataset. The same 25 Mpc/h boxes exist not only as total
matter, but as **gas density, gas temperature, neutral hydrogen (HI), electron density, magnetic field,
stellar mass, and velocities** — all on the *same aligned grid*. Those fields *are* the void contents (the
diffuse IGM, the warm-hot WHIM, the cold HI). So enriching the map is just adding layers.

The extensibility mechanism is a **field registry**: every layer is one dictionary entry tagged with a role
— `web` (classify the web from it), `input` (feed it to the network), or `target` (have the network
reconstruct it). Add a new survey product later → add one entry. The data loader and the model rebuild
themselves around the registry. This directly addresses the gap Bromley & Geller (2024) name in *Cosmology
with voids*: the full distribution of mass and gas inside voids "is not currently observable" — so we learn
to reconstruct it from the sparse tracers that *are*.

## 0. The field registry — the one place you edit to add a layer

In [ ]:
# torch already in env
!pip install torch

In [ ]:
import os, time, copy, urllib.request, numpy as np
import torch, torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm, LogNorm
from numpy.fft import fftn, ifftn, fftfreq
torch.manual_seed(0); np.random.seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

BASE = "https://users.flatironinstitute.org/~camels/CMD/3D_grids/data/IllustrisTNG"
def cmd_url(field): return f"{BASE}/Grids_{field}_IllustrisTNG_CV_128_z=0.0.npy"

# ============================================================================
#  FIELD REGISTRY  —  add a row to add a layer. Roles:
#   'web'    : compute the T-web (void/sheet/filament/knot) from this field
#   'input'  : stack as a network input channel (what you "observe")
#   'target' : the network reconstructs this  (what's hidden in the voids)
#  'log': apply log10(1+x) before use (for fields with huge dynamic range)
# ============================================================================
FIELDS = {
  'Mtot' : dict(file='tng_Mtot.npy', url=cmd_url('Mtot'), role='web'),
  'Mgas' : dict(file='tng_Mgas.npy', url=cmd_url('Mgas'), role='target', log=True, label='gas density'),
  'T'    : dict(file='tng_T.npy',    url=cmd_url('T'),    role='target', log=True, label='temperature'),
  'HI'   : dict(file='tng_HI.npy',   url=cmd_url('HI'),   role='target', log=True, label='neutral H'),
  # --- to add a layer later, e.g. magnetic field or electron density: ---
  # 'B'  : dict(file='tng_B.npy',  url=cmd_url('B'),  role='target', log=True, label='magnetic field'),
  # 'ne' : dict(file='tng_ne.npy', url=cmd_url('ne'), role='target', log=True, label='electron density'),
}
# the observable input is a sparse Poisson sampling of the matter field (a "galaxy survey")
TRACERS_FROM = 'Mtot'

L, R_smooth, lam_th, nbar = 25.0, 1.0, 0.3, 3.0
WEB_NAMES  = ['void','sheet','filament','knot']
WEB_COLORS = ['#08306b','#6baed6','#fd8d3c','#cb181d']
CMAP=ListedColormap(WEB_COLORS); NORM=BoundaryNorm([-.5,.5,1.5,2.5,3.5],4)

for name,spec in FIELDS.items():
    if not os.path.exists(spec['file']):
        print(f"downloading {name} (~216 MB) ..."); urllib.request.urlretrieve(spec['url'], spec['file'])
print("device:", device, "| registry roles:", {k:v['role'] for k,v in FIELDS.items()})

## 1. The multifield data layer

The web is computed from the `web` field; the observable input is a sparse tracer sampling of the matter;
the `target` fields are the contents the network will learn to reconstruct.

In [ ]:
def smooth(d,L,R):
    N=d.shape[0]; kax=fftfreq(N,d=L/N)*2*np.pi; KX,KY,KZ=np.meshgrid(kax,kax,kax,indexing='ij')
    return ifftn(fftn(d)*np.exp(-0.5*(KX**2+KY**2+KZ**2)*R**2)).real
def tweb(d,L,lam):
    N=d.shape[0]; kax=fftfreq(N,d=L/N)*2*np.pi; KX,KY,KZ=np.meshgrid(kax,kax,kax,indexing='ij')
    Kv=[KX,KY,KZ]; k2=KX**2+KY**2+KZ**2; k2[0,0,0]=1.0; dk=fftn(d); T=np.zeros((N,N,N,3,3))
    for i in range(3):
        for j in range(i,3):
            t=ifftn((Kv[i]*Kv[j]/k2)*dk).real; T[...,i,j]=t
            if i!=j: T[...,j,i]=t
    return np.sum(np.linalg.eigvalsh(T)>lam,-1).astype(np.int8)

def load_grid(field, gi):
    return np.array(np.load(FIELDS[field]['file'], mmap_mode='r')[gi], dtype=np.float64)

WEB_FIELD = [k for k,v in FIELDS.items() if v['role']=='web'][0]
def build_realization(gi, seed):
    # observable input channels, web labels, and target fields for one realization
    matter = load_grid(WEB_FIELD, gi)
    delta = matter/matter.mean() - 1.0
    web   = tweb(smooth(delta, L, R_smooth), L, lam_th)
    # --- observable input: sparse tracers of the matter field ---
    counts = np.random.default_rng(seed).poisson(nbar*np.clip(1+delta,0,None))
    inp = [np.log10(1.0+counts).astype(np.float32)]
    # --- targets: the content fields (log-compressed) ---
    targets = {}
    for name,spec in FIELDS.items():
        if spec['role']=='target':
            g = load_grid(name, gi)
            targets[name] = (np.log10(1.0+g) if spec.get('log') else g).astype(np.float32)
    return np.stack(inp), web, targets

TARGET_NAMES = [k for k,v in FIELDS.items() if v['role']=='target']
print("input channels: 1 (sparse tracers) | targets:", TARGET_NAMES)

## 2. What's in the voids (measured)

Per web class, on a real realization: gas density relative to the cosmic mean, median temperature, and the
share of all neutral hydrogen. Voids are underdense but not empty; the WHIM lives in filaments and knots.

In [ ]:
gi=0
matter=load_grid('Mtot',gi); web=tweb(smooth(matter/matter.mean()-1,L,R_smooth),L,lam_th)
Mgas=load_grid('Mgas',gi); Temp=load_grid('T',gi); HI=load_grid('HI',gi)
print(f"{'class':9s}{'vol%':>7s}{'gas/<gas>':>11s}{'median T (K)':>14s}{'HI share':>10s}")
for c in range(4):
    m=(web==c)
    print(f"{WEB_NAMES[c]:9s}{100*m.mean():7.1f}{np.median(Mgas[m])/Mgas.mean():11.3f}"
          f"{np.median(Temp[m]):14.3e}{100*HI[m].sum()/HI.sum():9.1f}%")

z=64
fig,ax=plt.subplots(1,4,figsize=(19,5))
im0=ax[0].imshow(web[z].T,origin='lower',cmap=CMAP,norm=NORM,extent=[0,L,0,L]); ax[0].set_title('cosmic web (from matter)')
cb=plt.colorbar(im0,ax=ax[0],ticks=[0,1,2,3],fraction=.046); cb.ax.set_yticklabels(WEB_NAMES)
im1=ax[1].imshow(Mgas[z].T,origin='lower',cmap='viridis',norm=LogNorm(vmin=Mgas.min(),vmax=Mgas.max()),extent=[0,L,0,L]); ax[1].set_title('gas density'); plt.colorbar(im1,ax=ax[1],fraction=.046)
im2=ax[2].imshow(Temp[z].T,origin='lower',cmap='hot',norm=LogNorm(vmin=1e3,vmax=Temp.max()),extent=[0,L,0,L]); ax[2].set_title('gas temperature (K)'); plt.colorbar(im2,ax=ax[2],fraction=.046)
im3=ax[3].imshow(HI[z].T,origin='lower',cmap='bone',norm=LogNorm(vmin=1e0,vmax=HI.max()),extent=[0,L,0,L]); ax[3].set_title('neutral hydrogen (HI)'); plt.colorbar(im3,ax=ax[3],fraction=.046)
for a in ax: a.set_xlabel('Mpc/h')
plt.tight_layout(); plt.show()

## 3. The extensible model — reconstruct the contents from sparse tracers

A shared 3D U-Net backbone with **one head per registry job**: a segmentation head for the web, and a
regression head for each `target` field. The heads are built *from the registry*, so adding a `target` adds
a head automatically. Input is the sparse tracer field — what a survey sees — and the network predicts the
web class and the hidden gas / temperature / HI together.

In [ ]:
def double_conv(ci,co):
    return nn.Sequential(nn.Conv3d(ci,co,3,padding=1),nn.BatchNorm3d(co),nn.ReLU(True),
                         nn.Conv3d(co,co,3,padding=1),nn.BatchNorm3d(co),nn.ReLU(True))
class MultiFieldUNet(nn.Module):
    # shared U-Net backbone; one seg head + one regression head per target (registry-driven)
    def __init__(self, in_ch, target_names, n_web=4, f=16):
        super().__init__()
        self.e1=double_conv(in_ch,f); self.e2=double_conv(f,2*f); self.b=double_conv(2*f,4*f)
        self.u2=nn.ConvTranspose3d(4*f,2*f,2,2); self.d2=double_conv(4*f,2*f)
        self.u1=nn.ConvTranspose3d(2*f,f,2,2);   self.d1=double_conv(2*f,f); self.p=nn.MaxPool3d(2)
        self.web_head = nn.Conv3d(f, n_web, 1)                                   # classify the web
        self.reg_heads = nn.ModuleDict({n: nn.Conv3d(f,1,1) for n in target_names})  # reconstruct each field
    def forward(self,x):
        e1=self.e1(x); e2=self.e2(self.p(e1)); b=self.b(self.p(e2))
        d2=self.d2(torch.cat([self.u2(b),e2],1)); feat=self.d1(torch.cat([self.u1(d2),e1],1))
        return self.web_head(feat), {n:h(feat) for n,h in self.reg_heads.items()}

def patchify(arr, p, s):
    N=arr.shape[-1]; out=[]
    for i in range(0,N-p+1,s):
        for j in range(0,N-p+1,s):
            for k in range(0,N-p+1,s): out.append(arr[...,i:i+p,j:j+p,k:k+p])
    return np.array(out)

patch, stride = 32, 32
TRAIN_IDS, VAL_IDS = [0,1], [2]
def assemble(ids, seed0):
    Xi,Yw,Yt = [],[],{n:[] for n in TARGET_NAMES}
    for gi in ids:
        inp,web,targets = build_realization(gi, seed0+gi)
        Xi.append(patchify(inp,patch,stride)); Yw.append(patchify(web,patch,stride))
        for n in TARGET_NAMES: Yt[n].append(patchify(targets[n],patch,stride))
    Xi=np.concatenate(Xi); Yw=np.concatenate(Yw)
    Yt={n:np.concatenate(v) for n,v in Yt.items()}
    return Xi,Yw,Yt

t0=time.time(); Xtr,Ywtr,Yttr=assemble(TRAIN_IDS,100); Xva,Ywva,Ytva=assemble(VAL_IDS,200)
# standardize input + each target with train stats (store to invert later)
xm,xs=Xtr.mean(),Xtr.std(); Xtr=(Xtr-xm)/xs; Xva=(Xva-xm)/xs
tstat={n:(Yttr[n].mean(),Yttr[n].std()+1e-6) for n in TARGET_NAMES}
for n in TARGET_NAMES:
    m,s=tstat[n]; Yttr[n]=(Yttr[n]-m)/s; Ytva[n]=(Ytva[n]-m)/s
print(f"assembled in {time.time()-t0:.0f}s | train {Xtr.shape} | targets {TARGET_NAMES}")

In [ ]:
net=MultiFieldUNet(Xtr.shape[1], TARGET_NAMES, f=16).to(device)
opt=torch.optim.Adam(net.parameters(),1e-3)
fr=np.bincount(Ywtr.ravel(),minlength=4); w=1/np.sqrt(fr+1); w=(w/w.sum()*4).astype(np.float32)
seg_loss=nn.CrossEntropyLoss(weight=torch.tensor(w).to(device)); reg_loss=nn.MSELoss()
print(f"{sum(p.numel() for p in net.parameters())} params | heads: web + {TARGET_NAMES}")

def batches(n,bs):
    idx=np.random.permutation(n)
    for i in range(0,n-bs+1,bs): yield idx[i:i+bs]
Xt=torch.tensor(Xtr); Wt=torch.tensor(Ywtr.astype(np.int64))
Tt={n:torch.tensor(Yttr[n][:,None]) for n in TARGET_NAMES}
bs,epochs=4,4; t0=time.time()
for ep in range(epochs):
    net.train()
    for b in batches(len(Xt),bs):
        xb=Xt[b].to(device); wb=Wt[b].to(device)
        opt.zero_grad(); seg,reg=net(xb)
        loss=seg_loss(seg,wb)+sum(reg_loss(reg[n],Tt[n][b].to(device)) for n in TARGET_NAMES)
        loss.backward(); opt.step()
    print(f"epoch {ep+1}/{epochs}  loss {loss.item():.3f}  ({time.time()-t0:.0f}s)")

## 4. See the reconstruction

From the sparse tracer input alone, the network predicts the web class **and** the hidden fields. Shown for
one validation slice: the web (true vs predicted) and each reconstructed content field (true vs predicted,
in log units).

In [ ]:
net.eval()
with torch.no_grad():
    seg,reg=net(torch.tensor(Xva[:1]).to(device))
    pweb=seg.argmax(1)[0].cpu().numpy()
zc=patch//2
ncol=1+1+len(TARGET_NAMES)
fig,ax=plt.subplots(2,ncol,figsize=(3.2*ncol,6.4))
ax[0,0].imshow(Xva[0,0,zc].T,origin='lower',cmap='inferno'); ax[0,0].set_title('input: sparse tracers'); ax[1,0].axis('off')
ax[0,1].imshow(Ywva[0,zc].T,origin='lower',cmap=CMAP,norm=NORM); ax[0,1].set_title('web — true')
ax[1,1].imshow(pweb[zc].T,origin='lower',cmap=CMAP,norm=NORM); ax[1,1].set_title('web — predicted')
for c,n in enumerate(TARGET_NAMES):
    m,s=tstat[n]
    true=Ytva[n][0,zc].T*s+m; pred=reg[n][0,0,zc].cpu().numpy().T*s+m
    vmin,vmax=true.min(),true.max()
    ax[0,2+c].imshow(true,origin='lower',cmap='magma',vmin=vmin,vmax=vmax); ax[0,2+c].set_title(f"{FIELDS[n].get('label',n)} — true")
    ax[1,2+c].imshow(pred,origin='lower',cmap='magma',vmin=vmin,vmax=vmax); ax[1,2+c].set_title('predicted')
for a in ax.ravel():
    if a.has_data(): a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()
print("From sparse galaxy-like tracers, the model reconstructs the web AND the diffuse gas/temperature/HI "
      "that aren't directly observable — exactly the 'hidden contents' Bromley & Geller flag.")

## How to layer in new data, and why it matters

**To add a layer:** add one row to `FIELDS` — a name, its URL (or a path to your own gridded survey map), and
a role. `role='target'` and the network grows a head to reconstruct it; `role='input'` and it becomes another
observed channel; `role='web'` swaps what defines the skeleton. Nothing else changes — the data loader and the
model rebuild around the registry. The two commented rows (magnetic field `B`, electron density `ne`) are
live examples; uncomment to pull and predict them.

**Why this is the right shape for new data.** The science is moving toward exactly these layers:
- **21cm / HI surveys** (e.g. SKA-era) will map neutral hydrogen across the web — drop in as an `HI` target or, once observed, an `input`.
- **X-ray and SZ** trace the **WHIM** and hot gas (the temperature/electron-density/pressure fields here) — the warm-hot baryons long "missing" from censuses.
- **Spectroscopic void-galaxy surveys** sharpen the tracer field itself.
- **Velocity reconstructions** (`Vcdm`, `Vgas`) connect to Bromley & Geller's outflow-based cosmology — the void's expansion rate encodes H₀, Ω_M, Ω_Λ — and to the "are we in a giant void?" question raised by the local-Hubble tension.

Each is one registry entry. The map stops being a static skeleton and becomes a living, multi-layer
reconstruction that gets more complete every time a new dataset arrives — predicting what's inside the voids
from what we can actually see.